# Silver Layer Cleaning

This notebook reads Bronze Delta tables, cleans the data, and writes Silver Delta tables.

# Silver Layer - Weather Data Cleaning

This notebook reads the Bronze Delta weather data,
performs data cleansing and validation,
and writes the cleaned data to the Silver layer

In [0]:
# Configure ADLS access

spark.conf.set(
    "fs.azure.account.key.@storageaccount.dfs.core.windows.net",
    "Access token"
)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql import functions as F

In [0]:
from pyspark.sql.functions import (
    col,
    trim,
    to_date,
    when,
    count,
    isnan
)

In [0]:
# ============================================================
# WEATHER - BRONZE TO SILVER
# ============================================================

weather_bronze_path = "abfss://processed@smartagrinil.dfs.core.windows.net/bronze/weather"
weather_silver_path = "abfss://processed@smartagrinil.dfs.core.windows.net/silver/weather"

weather = (
    spark.read
    .format("delta")
    .load(weather_bronze_path)
)

print("Before cleaning:", weather.count())
weather.printSchema()

# Remove duplicates
weather = weather.dropDuplicates()

# Remove completely empty rows
weather = weather.dropna(how="all")

# Trim string columns
for c in weather.columns:
    if dict(weather.dtypes)[c] == "string":
        weather = weather.withColumn(c, trim(col(c)))

# Fill missing numeric values with average
weather = weather.fillna({
    "temperature_c": weather.select(avg("temperature_c")).first()[0],
    "relative_humidity_pct": weather.select(avg("relative_humidity_pct")).first()[0]
})

# Remove negative values where not allowed
weather = weather.filter(
    (col("temperature_c") > -20) &
    (col("precipitation_mm") >= 0) &
    (col("relative_humidity_pct") >= 0)
)

print("After cleaning:", weather.count())

display(weather)

# Write Silver
weather.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(weather_silver_path)

print("Weather Silver created successfully!")

Before cleaning: 1520
root
 |-- date: date (nullable = true)
 |-- district: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- temperature_c: double (nullable = true)
 |-- precipitation_mm: double (nullable = true)
 |-- relative_humidity_pct: double (nullable = true)
 |-- solar_radiation_kwh_m2: double (nullable = true)

After cleaning: 1500


date,district,latitude,longitude,temperature_c,precipitation_mm,relative_humidity_pct,solar_radiation_kwh_m2
2025-06-29,Ahmednagar,20.3653,74.9402,27.79,8.36,67.23,5.55
2025-10-02,Nashik,17.5885,75.0731,33.49,31.12,38.79,6.52
2025-09-12,Pune,20.0319,73.8951,26.48,8.91,50.61,5.53
2025-04-02,Satara,17.4922,75.5414,24.61,11.89,72.43,8.64
2025-05-13,Pune,19.1196,75.8778,29.75,34.59,78.19,5.52
2025-02-18,Solapur,17.7103,74.831,30.27,2.67,66.86,5.77
2025-05-09,Aurangabad,20.4962,76.2484,26.54,5.15,79.81,4.23
2025-01-18,Solapur,19.7477,75.7211,22.08,5.3,66.81,2.9
2025-07-11,Pune,20.7357,75.0638,31.4,12.18,60.76,5.97
2025-10-09,Sangli,18.9902,76.4216,30.82,32.49,36.56,4.97


Weather Silver created successfully!


In [0]:
# ============================================================
# SOIL - BRONZE TO SILVER
# ============================================================

soil_bronze_path = "abfss://processed@smartagrinil.dfs.core.windows.net/bronze/soil"
soil_silver_path = "abfss://processed@smartagrinil.dfs.core.windows.net/silver/soil"

soil = (
    spark.read
    .format("delta")
    .load(soil_bronze_path)
)

print("Before cleaning:", soil.count())
soil.printSchema()

# Remove duplicates
soil = soil.dropDuplicates()

# Trim all string columns
for c in soil.columns:
    if dict(soil.dtypes)[c] == "string":
        soil = soil.withColumn(c, trim(col(c)))

# Remove rows where all values are null
soil = soil.dropna(how="all")

# Remove negative numeric values
numeric_columns = [
    field.name
    for field in soil.schema.fields
    if str(field.dataType) in [
        "IntegerType",
        "LongType",
        "DoubleType",
        "FloatType"
    ]
]

for c in numeric_columns:
    soil = soil.filter(
        col(c).isNull() | (col(c) >= 0)
    )

# Remove critical nulls if these columns exist
critical_soil = [
    c for c in ["district", "soil_type"]
    if c in soil.columns
]

if critical_soil:
    soil = soil.dropna(subset=critical_soil)

print("After cleaning:", soil.count())

display(soil)

# Write Silver
soil.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(soil_silver_path)

print("Soil Silver created successfully!")

Before cleaning: 1520
root
 |-- sample_date: date (nullable = true)
 |-- state: string (nullable = true)
 |-- district: string (nullable = true)
 |-- soil_type: string (nullable = true)
 |-- ph: double (nullable = true)
 |-- organic_carbon_pct: double (nullable = true)
 |-- nitrogen_kg_ha: double (nullable = true)
 |-- phosphorus_kg_ha: double (nullable = true)
 |-- potassium_kg_ha: double (nullable = true)
 |-- moisture_pct: double (nullable = true)

After cleaning: 1500


sample_date,state,district,soil_type,ph,organic_carbon_pct,nitrogen_kg_ha,phosphorus_kg_ha,potassium_kg_ha,moisture_pct
2025-07-29,Maharashtra,Solapur,Loamy,6.58,1.09,193.01,33.14,214.12,38.5
2024-09-30,Maharashtra,Ahmednagar,Alluvial,6.55,0.52,202.04,31.99,385.11,51.38
2024-01-30,Maharashtra,Sangli,Black,7.33,0.87,278.54,29.95,427.07,38.28
2025-07-15,Maharashtra,Dhule,Red,6.1,0.54,264.21,25.77,439.78,51.37
2025-12-22,Maharashtra,Solapur,Loamy,6.21,0.8,113.08,23.15,281.4,45.76
2025-05-08,Maharashtra,Nashik,Loamy,6.21,0.87,270.01,28.71,297.68,46.64
2024-04-13,Maharashtra,Aurangabad,Black,6.56,0.64,287.86,32.39,479.88,38.68
2025-07-10,Maharashtra,Dhule,Red,6.67,0.67,148.13,24.03,336.03,45.0
2025-09-11,Maharashtra,Nashik,Red,6.9,0.89,159.28,11.08,306.83,38.3
2024-12-29,Maharashtra,Nashik,Loamy,5.56,0.93,197.58,9.64,311.31,27.05


Soil Silver created successfully!


In [0]:
# ============================================================
# CROP YIELD - BRONZE TO SILVER
# ============================================================

crop_bronze_path = "abfss://processed@smartagrinil.dfs.core.windows.net/bronze/crop_yield"
crop_silver_path = "abfss://processed@smartagrinil.dfs.core.windows.net/silver/crop_yield"

crop = (
    spark.read
    .format("delta")
    .load(crop_bronze_path)
)

print("Before cleaning:", crop.count())
crop.printSchema()

# Remove duplicates
crop = crop.dropDuplicates()

# Remove completely empty rows
crop = crop.dropna(how="all")

# Trim string columns
for c in crop.columns:
    if dict(crop.dtypes)[c] == "string":
        crop = crop.withColumn(c, trim(col(c)))

# Remove negative numeric values
numeric_columns = [
    field.name
    for field in crop.schema.fields
    if str(field.dataType) in [
        "IntegerType",
        "LongType",
        "DoubleType",
        "FloatType"
    ]
]

for c in numeric_columns:
    crop = crop.filter(
        col(c).isNull() | (col(c) >= 0)
    )

print("After cleaning:", crop.count())

display(crop)

# Write Silver
crop.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(crop_silver_path)

print("Crop Yield Silver created successfully!")

Before cleaning: 1520
root
 |-- state: string (nullable = true)
 |-- district: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- crop: string (nullable = true)
 |-- area_harvested_ha: double (nullable = true)
 |-- production_tonnes: double (nullable = true)
 |-- yield_kg_per_ha: double (nullable = true)

After cleaning: 1500


state,district,year,crop,area_harvested_ha,production_tonnes,yield_kg_per_ha
Maharashtra,Kolhapur,2019,Cotton,38694.37,100286.28,1929.84
Maharashtra,Aurangabad,2025,Onion,25845.09,855314.08,4997.74
Maharashtra,Sangli,2024,Potato,9481.7,643597.21,1624.22
Maharashtra,Satara,2021,Potato,19600.73,223004.57,2112.25
Maharashtra,Solapur,2022,Wheat,48810.99,1220023.79,1875.83
Maharashtra,Aurangabad,2021,Soybean,2607.89,1100327.77,4617.86
Maharashtra,Sangli,2021,Tomato,3977.53,316805.41,4542.14
Maharashtra,Nashik,2020,Maize,8536.76,188007.15,5424.66
Maharashtra,Aurangabad,2022,Onion,37759.82,124741.76,2814.6
Maharashtra,Aurangabad,2018,Chickpea,20176.58,1295547.9,556.97


Crop Yield Silver created successfully!


In [0]:
# ============================================================
# MARKET PRICE - BRONZE TO SILVER
# ============================================================

market_bronze_path = "abfss://processed@smartagrinil.dfs.core.windows.net/bronze/market_price"
market_silver_path = "abfss://processed@smartagrinil.dfs.core.windows.net/silver/market_price"

market = (
    spark.read
    .format("delta")
    .load(market_bronze_path)
)

print("Before cleaning:", market.count())
market.printSchema()

# Remove duplicates
market = market.dropDuplicates()

# Remove completely empty rows
market = market.dropna(how="all")

# Trim string columns
for c in market.columns:
    if dict(market.dtypes)[c] == "string":
        market = market.withColumn(c, trim(col(c)))

# Remove negative numeric values
numeric_columns = [
    field.name
    for field in market.schema.fields
    if str(field.dataType) in [
        "IntegerType",
        "LongType",
        "DoubleType",
        "FloatType"
    ]
]

for c in numeric_columns:
    market = market.filter(
        col(c).isNull() | (col(c) >= 0)
    )

print("After cleaning:", market.count())

display(market)

# Write Silver
market.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(market_silver_path)

print("Market Price Silver created successfully!")

Before cleaning: 1520
root
 |-- arrival_date: date (nullable = true)
 |-- state: string (nullable = true)
 |-- district: string (nullable = true)
 |-- market: string (nullable = true)
 |-- commodity: string (nullable = true)
 |-- variety: string (nullable = true)
 |-- min_price_per_quintal: double (nullable = true)
 |-- max_price_per_quintal: double (nullable = true)
 |-- modal_price_per_quintal: double (nullable = true)

After cleaning: 1500


arrival_date,state,district,market,commodity,variety,min_price_per_quintal,max_price_per_quintal,modal_price_per_quintal
2025-08-04,Maharashtra,Aurangabad,Ahmednagar,Potato,Local,2852.05,3992.35,3326.29
2025-11-14,Maharashtra,Solapur,Lasalgaon,Cotton,Long Staple,447.14,1433.8,672.02
2025-07-11,Maharashtra,Sangli,Pune,Tomato,Hybrid,511.52,1151.32,1025.51
2025-01-01,Maharashtra,Pune,Nashik,Maize,Yellow,1877.63,2432.96,2162.15
2025-10-17,Maharashtra,Nashik,Lasalgaon,Soybean,Yellow,2366.84,3340.67,2655.01
2025-08-23,Maharashtra,Jalgaon,Sangli,Soybean,Bold,400.3,1594.38,718.54
2025-06-02,Maharashtra,Dhule,Pune,Maize,White,3734.47,4782.47,null
2025-06-15,Maharashtra,Jalgaon,Nashik,Potato,Jyoti,3625.01,4642.98,4152.72
2025-10-08,Maharashtra,Solapur,Ahmednagar,Wheat,Sharbati,2147.38,2652.72,2340.13
2025-04-07,Maharashtra,Satara,Jalgaon,Wheat,Sharbati,1585.64,2879.12,2062.57


Market Price Silver created successfully!


In [0]:
# ============================================================
# FINAL SILVER VERIFICATION
# ============================================================

weather_silver = spark.read.format("delta").load(weather_silver_path)
soil_silver = spark.read.format("delta").load(soil_silver_path)
crop_silver = spark.read.format("delta").load(crop_silver_path)
market_silver = spark.read.format("delta").load(market_silver_path)

print("======================================")
print("SILVER LAYER VERIFICATION")
print("======================================")

print("Weather rows:", weather_silver.count())
print("Soil rows:", soil_silver.count())
print("Crop Yield rows:", crop_silver.count())
print("Market Price rows:", market_silver.count())

print("======================================")
print("SILVER CLEANING COMPLETED!")
print("======================================")

SILVER LAYER VERIFICATION
Weather rows: 1480
Soil rows: 1500
Crop Yield rows: 1500
Market Price rows: 1500
SILVER CLEANING COMPLETED!
